In [16]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression

In [5]:
df = pd.read_csv('data_science_job.csv').drop(columns=['enrollee_id','city','city_development_index','enrolled_university','major_discipline','company_size'])

In [6]:
df.head()

,gender,relevent_experience,education_level,experience,company_type,training_hours,target
0,Male,Has relevent experience,Graduate,20.0,NaN,36.0,1.0
1,Male,No relevent experience,Graduate,15.0,Pvt Ltd,47.0,0.0
2,NaN,No relevent experience,Graduate,5.0,NaN,83.0,0.0
3,NaN,No relevent experience,Graduate,0.0,Pvt Ltd,52.0,1.0
4,Male,Has relevent experience,Masters,20.0,Funded Startup,8.0,0.0


In [8]:
df.shape

(19158, 7)

In [7]:
df.isnull().sum()

gender                 4508
relevent_experience       0
education_level         460
experience               65
company_type           6140
training_hours          766
target                    0
dtype: int64

In [9]:
x = df.drop(columns=['target'])
y = df['target']

In [10]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=2)

In [23]:
numerical_features = ['experience','training_hours']
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = ['gender','relevent_experience','education_level','company_type']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',OrdinalEncoder())
])

In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [25]:
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

In [26]:
from sklearn import set_config

set_config(display='diagram')
clf

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['experience',
                                                   'training_hours']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OrdinalEncoder())]),
                                                  ['gender',
                                                   'relevent_experience',
                                                   'education_level',
                                                   'company_type'])])),
                ('classifier', LogisticRegression())])

In [27]:
para_grid = {
    'preprocessor__num__imputer__strategy':['mean','median'],
    'preprocessor__cat__imputer__strategy':['most_frequent','constant'],
    'classifier__C':[0.1,1,10,100]
}

grid_search = GridSearchCV(clf,para_grid,cv=10)

In [28]:
grid_search = grid_search.fit(x_train,y_train)

print(f'Best Paras:')
print(grid_search.best_params_)

Best Paras:
{'classifier__C': 1, 'preprocessor__cat__imputer__strategy': 'constant', 'preprocessor__num__imputer__strategy': 'mean'}


In [29]:
print(f"Internal CV score: {grid_search.best_score_:.3f}")

Internal CV score: 0.753


In [30]:
import pandas as pd

cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results = cv_results.sort_values("mean_test_score", ascending=False)
cv_results[['param_classifier__C','param_preprocessor__cat__imputer__strategy','param_preprocessor__num__imputer__strategy','mean_test_score']]

,param_classifier__C,param_preprocessor__cat__imputer__strategy,param_preprocessor__num__imputer__strategy,mean_test_score
6,1.0,constant,mean,0.752903
7,1.0,constant,median,0.752903
11,10.0,constant,median,0.752903
15,100.0,constant,median,0.752903
10,10.0,constant,mean,0.752773
14,100.0,constant,mean,0.752773
2,0.1,constant,mean,0.752708
3,0.1,constant,median,0.752512
0,0.1,most_frequent,mean,0.751207
1,0.1,most_frequent,median,0.751207
